# MNIST Digit Classification Project

This notebook presents a professional end-to-end workflow for digit recognition using the MNIST dataset. The project covers data loading, preprocessing, model design, training, evaluation, and visualization. The final model is designed to achieve above 90% test accuracy.


## 1. Setup and Dependencies

Import required libraries, set random seeds for reproducibility, and verify the execution environment.


In [3]:
import os
import random
import ssl
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Work around SSL certificate issues in some environments
ssl._create_default_https_context = ssl._create_unverified_context

seed = 42
os.environ["PYTHONHASHSEED"] = str(seed)
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)


TensorFlow version: 2.16.2
NumPy version: 1.26.4


## 2. Load and Explore Data

Load the MNIST dataset, inspect shapes, and display sample digits.


In [5]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print("Training set:", x_train.shape, y_train.shape)
print("Test set:", x_test.shape, y_test.shape)

values, counts = np.unique(y_train, return_counts=True)
for value, count in zip(values, counts):
    print(f"Digit {value}: {count} samples")

plt.figure(figsize=(10, 4))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()


Training set: (60000, 28, 28) (60000,)
Test set: (10000, 28, 28) (10000,)
Digit 0: 5923 samples
Digit 1: 6742 samples
Digit 2: 5958 samples
Digit 3: 6131 samples
Digit 4: 5842 samples
Digit 5: 5421 samples
Digit 6: 5918 samples
Digit 7: 6265 samples
Digit 8: 5851 samples
Digit 9: 5949 samples


<string>:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 3. Preprocessing

Normalize pixel values to [0, 1], create a validation split, and prepare the data for training.


In [7]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=0.20, random_state=seed, stratify=y_train
)

print("Train:", x_train.shape, y_train.shape)
print("Validation:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)


Train: (48000, 28, 28) (48000,)
Validation: (12000, 28, 28) (12000,)
Test: (10000, 28, 28) (10000,)


## 4. Model Architecture

Build a compact dense neural network with dropout and ReLU activation. This model is designed to generalize well on MNIST.


In [9]:
saud_model = keras.Sequential([
    keras.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(10, activation="softmax"),
])

saud_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

saud_model.summary()


Model: "sequential"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       200,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼─────────────────────

## 5. Training

Train the model with early stopping based on validation loss to maintain strong generalization.


In [11]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
]

history = saud_model.fit(
    x_train,
    y_train,
    epochs=25,
    batch_size=128,
    validation_data=(x_val, y_val),
    callbacks=callbacks,
    verbose=2,
)


Epoch 1/25
375/375 - 3s - 8ms/step - accuracy: 0.8870 - loss: 0.3758 - val_accuracy: 0.9506 - val_loss: 0.1683
Epoch 2/25
375/375 - 2s - 5ms/step - accuracy: 0.9534 - loss: 0.1564 - val_accuracy: 0.9646 - val_loss: 0.1182
Epoch 3/25
375/375 - 2s - 5ms/step - accuracy: 0.9648 - loss: 0.1127 - val_accuracy: 0.9693 - val_loss: 0.0989
Epoch 4/25
375/375 - 2s - 5ms/step - accuracy: 0.9726 - loss: 0.0880 - val_accuracy: 0.9722 - val_loss: 0.0916
Epoch 5/25
375/375 - 2s - 6ms/step - accuracy: 0.9775 - loss: 0.0719 - val_accuracy: 0.9758 - val_loss: 0.0813
Epoch 6/25
375/375 - 2s - 6ms/step - accuracy: 0.9797 - loss: 0.0616 - val_accuracy: 0.9783 - val_loss: 0.0767
Epoch 7/25
375/375 - 2s - 6ms/step - accuracy: 0.9835 - loss: 0.0524 - val_accuracy: 0.9776 - val_loss: 0.0742
Epoch 8/25
375/375 - 2s - 6ms/step - accuracy: 0.9857 - loss: 0.0463 - val_accuracy: 0.9785 - val_loss: 0.0784
Epoch 9/25
375/375 - 2s - 6ms/step - accuracy: 0.9875 - loss: 0.0396 - val_accuracy: 0.9791 - val_loss: 0.0797
E

## 6. Training Metrics

Visualize training and validation accuracy and loss to confirm stable learning.


In [13]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="train_accuracy")
plt.plot(history.history["val_accuracy"], label="val_accuracy")
plt.title("Accuracy over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="train_loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.title("Loss over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


<string>:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 7. Test Evaluation

Evaluate the final model on the held-out test set and display sample predictions.


In [15]:
test_loss, test_accuracy = saud_model.evaluate(x_test, y_test, verbose=2)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy * 100:.2f}%")

sample_indices = np.random.choice(len(x_test), size=12, replace=False)
plt.figure(figsize=(10, 6))
for i, idx in enumerate(sample_indices):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_test[idx], cmap="gray")
    prediction = np.argmax(saud_model.predict(x_test[idx:idx + 1]), axis=1)[0]
    plt.title(f"True: {y_test[idx]} / Pred: {prediction}")
    plt.axis("off")
plt.tight_layout()
plt.show()


313/313 - 0s - 2ms/step - accuracy: 0.9791 - loss: 0.0713
Test loss: 0.0713
Test accuracy: 97.91%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step


<string>:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 8. Summary

This notebook provides a clear project narrative and a professional model pipeline for MNIST digit classification. The model uses dropout and validation monitoring to deliver strong generalization performance above 90% accuracy.


## 8. Saud Model Summary

The project now trains and evaluates the dense neural network as `saud_model`.
This model uses two hidden dense layers with dropout, and the full pipeline includes: preprocessing, training with early stopping, and test evaluation.
The model achieves strong generalization on MNIST, exceeding 90% test accuracy.
